# Multi-Class Text Classification: Impact of Stopword Removal

This section describes the workflow and rationale for evaluating the impact of **Khmer stopword removal** on multi-class text classification. Both **raw** and **cleaned** versions of the dataset are tested to measure changes in model performance.

---

## Dataset Versions

| Version | Description |
|---------|-------------|
| Raw     | Original dataset with no stopword removal |
| Cleaned | Stopwords removed using the broad classes stopword list |

**Goal:** Determine how stopword removal affects classification metrics such as accuracy, F1-macro, precision, and recall.

---

## Models Tested

| Model                    | Library       | Reason for Use                                                      | Speed      |
|---------------------------|---------------|--------------------------------------------------------------------|------------|
| Multinomial Naive Bayes   | scikit-learn  | Classic baseline for text classification; works well with TF-IDF  | Very fast  |
| Logistic Regression       | scikit-learn  | Interpretable; strong baseline; works well on sparse TF-IDF data  | Fast       |
| Linear SVC                | scikit-learn  | Often slightly better than LR on sparse high-dimensional data     | Fast       |

- All models use **TF-IDF vectorization** with a Khmer-aware token pattern.  
- Bigrams `(1,2)` are included to capture short phrases.

---

## Workflow

1. Load train/test splits for both **raw** and **cleaned** versions.  
2. Define **TF-IDF settings** optimized for Khmer:
   - Max features: 12,000  
   - Minimum document frequency: 3  
   - Bigram inclusion  
   - Khmer Unicode token pattern
3. Train each model using a **pipeline**: `TF-IDF → Classifier`.  
4. Evaluate using:
   - Accuracy  
   - Macro-F1  
   - Macro-Precision  
   - Macro-Recall  
5. Save **per-class classification reports** for detailed analysis.  
6. Generate a **summary table** comparing metrics across versions.

---

## Evaluation and Comparison

- Metrics captured for each model:
  - Accuracy  
  - F1-macro  
  - Δ (delta) between cleaned vs raw versions

- **Interpretation guidance**:
  - **Δ Accuracy > 0** and **Δ F1 > 0** → stopword removal improved classification.  
  - **Δ < 0** → stopword list may remove meaningful domain-specific terms.  
  - Inspect per-class metrics to determine which categories benefit most (e.g., Politics, Sports, Technology).  
  - Training time is often faster on cleaned data due to reduced feature space.

---

## Expected Insights

1. **Stopword removal reduces noise**:
   - Functional words such as `និង`, `បាន`, `ដែល` no longer dominate TF-IDF features.  
2. **Content-bearing terms gain importance**:
   - Words like `កម្ពុជា`, `ក្រុមហ៊ុន`, `រថយន្ត` now influence classification more strongly.  
3. **Model performance may improve**:
   - Cleaned data often yields higher F1-macro and accuracy.  
   - Some classes may see significant gains, depending on the domain.  
4. **Pipeline reusability**:
   - Same workflow can be applied to future Khmer datasets with minimal changes.  
5. **Potential next steps**:
   - Visualize confusion matrices to identify misclassified classes.  
   - Analyze top TF-IDF terms per class before/after stopword removal.  
   - Optionally incorporate Khmer-specific embeddings (e.g., Khmer-BERT) for richer representation.

---

## Files and Outputs

| File / Directory | Description |
|-----------------|-------------|
| `class_report_*` | Per-model classification report with per-class metrics |
| `classification_comparison_summary.csv` | Summary table with raw vs cleaned metrics and Δ values |
| Plots directory | Optional visualization of performance and feature importance |

---

**Summary:**  
Training and evaluating models on both raw and cleaned datasets provides empirical evidence of how stopword removal affects text classification. Metrics such as accuracy and F1-macro directly reflect the quality of the TF-IDF representation and the stopword list.



In [2]:
# =============================================================================
# Multi-class Text Classification: Raw vs Stopword-removed
# Goal: Measure impact of Khmer stopword removal on classification performance
# =============================================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
from time import time

print("Starting classification comparison (Raw vs Cleaned)...\n")

# ─── Paths ──────────────────────────────────────────────────────────────────
processed_dir = Path(r"D:/Year 5/S1/Information Retrieval/StopwordProject/khmer_stopword/khmer_stopword_removal_system/data/processed")
output_dir = Path(r"D:/Year 5/S1/Information Retrieval/StopwordProject/khmer_stopword/khmer_stopword_removal_system/results/eda")
output_dir.mkdir(exist_ok=True)

# ─── Label names (used in classification report) ────────────────────────────
label_names = ["economic", "entertainment", "life", "politic", "sport", "technology"]

# ─── Load train & test ──────────────────────────────────────────────────────
def load_split(split_name, version='raw'):
    path = processed_dir / f"{split_name}_{version}.pkl"
    if not path.is_file():
        print(f"Missing file: {path}")
        return None, None, None
    df = pd.read_pickle(path)
    X = df['content'] if version == 'raw' else df['cleaned_content']
    y = df['label']
    print(f"Loaded {version} {split_name}: {len(X)} samples")
    return X, y, df

# ─── Load data ──────────────────────────────────────────────────────────────
X_train_raw, y_train_raw, _ = load_split('train', 'raw')
X_test_raw,  y_test_raw,  _ = load_split('test',  'raw')

X_train_clean, y_train_clean, _ = load_split('train', 'cleaned')
X_test_clean,  y_test_clean,  _ = load_split('test',  'cleaned')

if any(x is None for x in [X_train_raw, X_test_raw, X_train_clean, X_test_clean]):
    print("Cannot continue — missing train or test data")
else:
    print("\nAll splits loaded successfully.")

# ─── Define models to compare ───────────────────────────────────────────────
models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=300, random_state=42, n_jobs=-1),
    'Linear SVC': LinearSVC(max_iter=500, random_state=42)
}

# ─── TF-IDF settings (Khmer-aware) ──────────────────────────────────────────
tfidf_params = {
    'max_features': 12000,
    'ngram_range': (1, 2),
    'min_df': 3,
    'token_pattern': r'(?u)\b[\u1780-\u17FF\u200B]{2,}\b',
    'sublinear_tf': True,
}

# ─── Train & evaluate function ──────────────────────────────────────────────
def evaluate_pipeline(X_train, y_train, X_test, y_test, name_version):
    results = {}
    
    for model_name, clf in models.items():
        print(f"\n→ Training {model_name} on {name_version} data...")
        t0 = time()
        
        pipeline = Pipeline([
            ('tfidf', TfidfVectorizer(**tfidf_params)),
            ('clf', clf)
        ])
        
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        
        duration = time() - t0
        
        acc = accuracy_score(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
        prec_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
        rec_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)
        
        results[model_name] = {
            'accuracy': acc,
            'f1_macro': f1_macro,
            'precision_macro': prec_macro,
            'recall_macro': rec_macro,
            'time_sec': duration
        }
        
        print(f"  Accuracy:       {acc:.4f}")
        print(f"  F1-macro:       {f1_macro:.4f}")
        print(f"  Precision-macro:{prec_macro:.4f}")
        print(f"  Recall-macro:   {rec_macro:.4f}")
        print(f"  Time:           {duration:.1f} seconds")
        
        # Save classification report with correct label names
        try:
            report = classification_report(
                y_test,
                y_pred,
                target_names=label_names,
                output_dict=True,
                zero_division=0
            )
            report_df = pd.DataFrame(report).transpose()
            report_path = output_dir / f"class_report_{name_version}_{model_name.replace(' ', '_')}.csv"
            report_df.to_csv(report_path, encoding='utf-8-sig')
            print(f"  Saved detailed report → {report_path.name}")
        except Exception as e:
            print(f"  Warning: Could not generate classification report: {e}")
    
    return results

# ─── Run both versions ──────────────────────────────────────────────────────
print("\n" + "═"*70)
print("Evaluating RAW version (no stopword removal)")
raw_results = evaluate_pipeline(X_train_raw, y_train_raw, X_test_raw, y_test_raw, "raw")

print("\n" + "═"*70)
print("Evaluating CLEANED version (after stopword removal)")
clean_results = evaluate_pipeline(X_train_clean, y_train_clean, X_test_clean, y_test_clean, "cleaned")

# ─── Summary table ──────────────────────────────────────────────────────────
summary = []
for model in models:
    r = raw_results.get(model, {})
    c = clean_results.get(model, {})
    summary.append({
        'Model': model,
        'Acc Raw': f"{r.get('accuracy', 'N/A'):.4f}" if 'accuracy' in r else 'N/A',
        'Acc Clean': f"{c.get('accuracy', 'N/A'):.4f}" if 'accuracy' in c else 'N/A',
        'Δ Acc': f"{c.get('accuracy', 0) - r.get('accuracy', 0):+.4f}" if 'accuracy' in r and 'accuracy' in c else 'N/A',
        'F1 Raw': f"{r.get('f1_macro', 'N/A'):.4f}" if 'f1_macro' in r else 'N/A',
        'F1 Clean': f"{c.get('f1_macro', 'N/A'):.4f}" if 'f1_macro' in c else 'N/A',
        'Δ F1': f"{c.get('f1_macro', 0) - r.get('f1_macro', 0):+.4f}" if 'f1_macro' in r and 'f1_macro' in c else 'N/A',
    })

summary_df = pd.DataFrame(summary)
print("\n" + "═"*70)
print("Final Comparison Table:")
print(summary_df.to_string(index=False))

# Save summary
summary_path = output_dir / "classification_comparison_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')
print(f"\nSummary saved → {summary_path}")

print("\nDone. Check the following files in your results/eda folder:")
print("• class_report_raw_*.csv")
print("• class_report_cleaned_*.csv")
print("• classification_comparison_summary.csv")

Starting classification comparison (Raw vs Cleaned)...

Loaded raw train: 5140 samples
Loaded raw test: 1543 samples
Loaded cleaned train: 5140 samples
Loaded cleaned test: 1543 samples

All splits loaded successfully.

══════════════════════════════════════════════════════════════════════
Evaluating RAW version (no stopword removal)

→ Training Naive Bayes on raw data...
  Accuracy:       0.8756
  F1-macro:       0.8627
  Precision-macro:0.8754
  Recall-macro:   0.8578
  Time:           2.2 seconds
  Saved detailed report → class_report_raw_Naive_Bayes.csv

→ Training Logistic Regression on raw data...


d:\Year 5\S1\Information Retrieval\StopwordProject\khmer_stopword\khmer_stopword_removal_system\my_venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  Accuracy:       0.9177
  F1-macro:       0.9099
  Precision-macro:0.9129
  Recall-macro:   0.9072
  Time:           2.7 seconds
  Saved detailed report → class_report_raw_Logistic_Regression.csv

→ Training Linear SVC on raw data...
  Accuracy:       0.9255
  F1-macro:       0.9185
  Precision-macro:0.9188
  Recall-macro:   0.9183
  Time:           2.3 seconds
  Saved detailed report → class_report_raw_Linear_SVC.csv

══════════════════════════════════════════════════════════════════════
Evaluating CLEANED version (after stopword removal)

→ Training Naive Bayes on cleaned data...
  Accuracy:       0.8730
  F1-macro:       0.8614
  Precision-macro:0.8750
  Recall-macro:   0.8555
  Time:           1.8 seconds
  Saved detailed report → class_report_cleaned_Naive_Bayes.csv

→ Training Logistic Regression on cleaned data...


d:\Year 5\S1\Information Retrieval\StopwordProject\khmer_stopword\khmer_stopword_removal_system\my_venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  Accuracy:       0.9190
  F1-macro:       0.9124
  Precision-macro:0.9157
  Recall-macro:   0.9093
  Time:           2.1 seconds
  Saved detailed report → class_report_cleaned_Logistic_Regression.csv

→ Training Linear SVC on cleaned data...
  Accuracy:       0.9294
  F1-macro:       0.9232
  Precision-macro:0.9234
  Recall-macro:   0.9231
  Time:           2.0 seconds
  Saved detailed report → class_report_cleaned_Linear_SVC.csv

══════════════════════════════════════════════════════════════════════
Final Comparison Table:
              Model Acc Raw Acc Clean   Δ Acc F1 Raw F1 Clean    Δ F1
        Naive Bayes  0.8756    0.8730 -0.0026 0.8627   0.8614 -0.0013
Logistic Regression  0.9177    0.9190 +0.0013 0.9099   0.9124 +0.0025
         Linear SVC  0.9255    0.9294 +0.0039 0.9185   0.9232 +0.0047

Summary saved → D:\Year 5\S1\Information Retrieval\StopwordProject\khmer_stopword\khmer_stopword_removal_system\results\eda\classification_comparison_summary.csv

Done. Check the following

# Multi-Class Classification Results: Raw vs Cleaned Data

This section interprets the results of training three classifiers (Naive Bayes, Logistic Regression, Linear SVC) on the **raw dataset** (no stopword removal) and the **cleaned dataset** (stopwords removed).

---

## Dataset Summary

| Version  | Train Samples | Test Samples |
|----------|---------------|--------------|
| Raw      | 5,140         | 1,543        |
| Cleaned  | 5,140         | 1,543        |

All splits were successfully loaded, ensuring a fair comparison between raw and cleaned datasets.

---

## Performance Metrics

The following metrics were computed for each model:

- **Accuracy**: overall fraction of correctly predicted labels  
- **F1-macro**: macro-averaged F1 score across all classes  
- **Precision-macro**: macro-averaged precision across all classes  
- **Recall-macro**: macro-averaged recall across all classes  
- **Time**: training and evaluation duration (seconds)  

---

## Model Performance Comparison

| Model                  | Accuracy Raw | Accuracy Clean | Δ Accuracy | F1 Raw  | F1 Clean | Δ F1    |
|------------------------|--------------|----------------|------------|---------|----------|---------|
| Naive Bayes            | 0.8756       | 0.8730         | -0.0026    | 0.8627  | 0.8614   | -0.0013 |
| Logistic Regression    | 0.9177       | 0.9190         | +0.0013    | 0.9099  | 0.9124   | +0.0025 |
| Linear SVC             | 0.9255       | 0.9294         | +0.0039    | 0.9185  | 0.9232   | +0.0047 |

**Δ values** show the difference between the cleaned and raw datasets (Cleaned − Raw). Positive Δ indicates performance improvement after stopword removal.

---

## Observations

1. **Naive Bayes**
   - Slight decrease in accuracy (-0.26%) and F1-macro (-0.13%) after stopword removal.  
   - Indicates that some stopwords may contribute slightly to predictive power in this model, possibly due to its dependence on raw term frequencies.

2. **Logistic Regression**
   - Small improvement in accuracy (+0.13%) and F1-macro (+0.25%) with stopword removal.  
   - Suggests cleaner features helped model weights better discriminate between classes.

3. **Linear SVC**
   - Largest improvement among the three models: Δ Accuracy +0.39%, Δ F1 +0.47%.  
   - Indicates that removing stopwords improved separation in the high-dimensional sparse feature space, benefiting linear classifiers the most.

---

## Insights

- Stopword removal has **model-dependent effects**:
  - Simple frequency-based models (Naive Bayes) may see minimal or slightly negative impact.
  - Weight-based models (Logistic Regression, Linear SVC) benefit from cleaner, more informative features.

- **Training time** slightly decreased on cleaned data:
  - Fewer features in the TF-IDF matrix reduce computation time (e.g., Linear SVC trained 0.3s faster).

- Overall, stopword removal **improves discriminative power** for stronger classifiers while having minimal negative impact on simpler models.

- Per-class classification reports (`class_report_*.csv`) can be used to:
  - Identify which categories gain most from stopword removal.  
  - Analyze misclassifications for potential refinement of stopword lists.

---

## Recommendations

1. Keep the **broad-classes stopword list** for TF-IDF-based classification, as it improves performance for Logistic Regression and Linear SVC.  
2. Explore **per-class feature importance** to further refine the stopword list.  
3. Optionally, test **Khmer embeddings** (e.g., Khmer-BERT) for comparison with TF-IDF results.  
4. Document Δ Accuracy and Δ F1 values in the project report to support the discussion on stopword removal impact.

---

**Conclusion:**  

Stopword removal generally enhances the performance of linear classifiers on Khmer text, demonstrating that **feature cleaning improves class discrimination**. The effect is modest for Naive Bayes but noticeable for Logistic Regression and Linear SVC.
